In [1]:
import os
import numpy
import utils
import logging

### 设置日志

In [23]:
# 配置日志格式和级别
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    filename='./overlap.log',
                    filemode='w')
# 创建控制台输出
console = logging.StreamHandler()
console.setLevel(logging.INFO)
formatter = logging.Formatter('%(levelname)s - %(message)s')
console.setFormatter(formatter)
logging.getLogger('').addHandler(console)

### 加载各模版长度（index len）

In [2]:
template_len = {}
template_dir = './templates/'
template_files = [f for f in os.listdir(template_dir) if f.endswith('.h5')]
for template_file in template_files:
    try:
        tmp = len(utils.load_template(template_file,template_dir)['waveforms'][0][0])
        template_len[template_file] = tmp
    except KeyError:
        print(f"Error: 'waveforms' key not found in {template_file}. Skipping...")
    except IndexError:
        print(f"Error: Unexpected structure in {template_file}. Skipping...")
    except Exception as e:
        print(f"Unexpected error processing {template_file}: {e}. Skipping...")

### 加载匹配事件结果

In [3]:
logging_dir = './logs_814-918/'
stations_dir = os.listdir(logging_dir)
stations_dir.sort()

In [31]:
def remove_overlapping_indices(indices, template_len):
    """
    删除所有与当前索引距离小于模板长度的后续索引。
    
    :param indices: 输入的索引列表
    :param template_len: 模板长度
    :return: 处理后的索引列表
    """
    if not indices:
        return []
    
    result = []
    i = 0
    
    while i < len(indices):
        current_index = indices[i]
        result.append(i)  # 保留当前索引
        
        # 找到所有需要跳过的索引
        j = i + 1
        while j < len(indices) and indices[j] - current_index < template_len:
            j += 1
        
        # 跳到下一个未被删除的索引
        i = j
    
    return result

In [32]:
for station_dir in stations_dir:
    logging.info(f'Processing station: {station_dir}')
    
    # 构造完整的目标目录路径
    target_dir = os.path.join('./logs_814-918_overlap', station_dir)
    os.makedirs(target_dir, exist_ok=True)  # 创建目标目录（如果已存在则不会报错）
    
    full_path = os.path.join(logging_dir, station_dir)
    result_files = [f for f in os.listdir(full_path) if f.endswith('.h5')]
    
    for result_file in result_files:
        logging.info(f'Reading result file: {result_file}')
        
        # 读取结果数据
        data = utils.load_result(result_file, path=os.path.join(full_path, ''))
        
        # 遍历每个模板文件并处理其匹配事件索引
        for template_file in template_files:
            template_file_len = template_len[template_file]
            
            # 获取 matched_event_index_all 和 matched_event_index_daily
            tmp_matched_event_index_all = data[template_file]['matched_event_index_all']
            tmp_matched_event_index_daily = data[template_file]['matched_event_index_daily']
            
            # 初始化需要删除的全局索引集合
            delete_idx_all = set()  # 存储需要从 matched_event_index_all 删除的索引
            
            # 处理每日数据
            filtered_daily_indices = []
            global_index_offset = 0  # 记录当前每日数据在全局数组中的起始位置
            
            for daily_indices in tmp_matched_event_index_daily:
                if not daily_indices:
                    # 如果每日数据为空，跳过处理
                    filtered_daily_indices.append([])
                    continue
                
                # 找到需要保留的索引位置
                local_keep_indices = remove_overlapping_indices(daily_indices, template_file_len)
                
                # 根据保留的索引位置过滤每日数据
                filtered_daily = [daily_indices[i] for i in local_keep_indices]
                filtered_daily_indices.append(filtered_daily)
                
                # 找到需要删除的局部索引位置（即未被保留的索引）
                local_delete_indices = set(range(len(daily_indices))) - set(local_keep_indices)
                
                # 将局部索引映射到全局索引
                for local_idx in local_delete_indices:
                    global_idx = global_index_offset + local_idx
                    delete_idx_all.add(global_idx)
                
                # 更新全局偏移量
                global_index_offset += len(daily_indices)
            
            # 更新 matched_event_index_daily
            data[template_file]['matched_event_index_daily'] = filtered_daily_indices
            
            # 统一更新 matched_event_index_all
            filtered_all_indices = [
                idx for i, idx in enumerate(tmp_matched_event_index_all) if i not in delete_idx_all
            ]
            data[template_file]['matched_event_index_all'] = filtered_all_indices
            
            # 更新 matched_count 和 matched_daily
            matched_count_reduction = len(delete_idx_all)
            data[template_file]['matched_count'] -= matched_count_reduction
        
        # 将修改后的数据保存到目标目录
        output_path = os.path.join(target_dir, result_file)
        utils.save_result(data, output_path)
        logging.info(f'Saved modified result {result_file} to: {output_path}')

INFO - Processing station: Xs01
INFO - Processing station: Xs01
INFO - Processing station: Xs01
INFO - Processing station: Xs01
INFO - Reading result file: Xs01_matched_filter_results_threshold0.8_weight0.0_0.0_1.0.h5
INFO - Reading result file: Xs01_matched_filter_results_threshold0.8_weight0.0_0.0_1.0.h5
INFO - Reading result file: Xs01_matched_filter_results_threshold0.8_weight0.0_0.0_1.0.h5
INFO - Reading result file: Xs01_matched_filter_results_threshold0.8_weight0.0_0.0_1.0.h5


INFO - Saved modified result Xs01_matched_filter_results_threshold0.8_weight0.0_0.0_1.0.h5 to: ./logs_814-918_overlap/Xs01/Xs01_matched_filter_results_threshold0.8_weight0.0_0.0_1.0.h5
INFO - Saved modified result Xs01_matched_filter_results_threshold0.8_weight0.0_0.0_1.0.h5 to: ./logs_814-918_overlap/Xs01/Xs01_matched_filter_results_threshold0.8_weight0.0_0.0_1.0.h5
INFO - Saved modified result Xs01_matched_filter_results_threshold0.8_weight0.0_0.0_1.0.h5 to: ./logs_814-918_overlap/Xs01/Xs01_matched_filter_results_threshold0.8_weight0.0_0.0_1.0.h5
INFO - Saved modified result Xs01_matched_filter_results_threshold0.8_weight0.0_0.0_1.0.h5 to: ./logs_814-918_overlap/Xs01/Xs01_matched_filter_results_threshold0.8_weight0.0_0.0_1.0.h5
INFO - Reading result file: Xs01_matched_filter_results_threshold0.7_weight0.0_1.0_0.0.h5
INFO - Reading result file: Xs01_matched_filter_results_threshold0.7_weight0.0_1.0_0.0.h5
INFO - Reading result file: Xs01_matched_filter_results_threshold0.7_weight0.0_1

### 验证处理是否有效

In [4]:
# 配置日志文件
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    filename='./overlap_verification.log',
                    filemode='w')
# 创建控制台输出
console = logging.StreamHandler()
console.setLevel(logging.INFO)
formatter = logging.Formatter('%(levelname)s - %(message)s')
console.setFormatter(formatter)
logging.getLogger('').addHandler(console)

In [5]:
def verify_overlap(template_len, stations_dir, target_dir='./logs_814-918_overlap'):
    """
    验证处理后的文件是否存在重叠，并将结果记录到日志文件中。
    """
    for station_dir in stations_dir:
        logging.info(f'Verifying station: {station_dir}')
        
        station_target_dir = os.path.join(station_dir,'')
        station_target_dir = os.path.join(target_dir, station_target_dir)
        result_files = [f for f in os.listdir(station_target_dir) if f.endswith('.h5')]
        
        for result_file in result_files:
            logging.info(f'Verifying result file: {result_file}')
            data = utils.load_result(result_file, path=station_target_dir)
            
            for template_file, template_file_len in template_len.items():
                matched_event_index_all = data[template_file]['matched_event_index_all']
                matched_event_index_daily = data[template_file]['matched_event_index_daily']
                
                # 验证 matched_event_index_all
                for i in range(len(matched_event_index_all) - 1):
                    current_index = matched_event_index_all[i]
                    next_index = matched_event_index_all[i + 1]
                    difference = next_index - current_index
                    
                    if difference < template_file_len:
                        logging.error(f'Overlap found in matched_event_index_all (template: {template_file}): '
                                      f'current_index={current_index}, next_index={next_index}, difference={difference}')
                
                # 验证 matched_event_index_daily
                for tmp_i, daily_indices in enumerate(matched_event_index_daily):
                    for j in range(len(daily_indices) - 1):
                        current_index = daily_indices[j]
                        next_index = daily_indices[j + 1]
                        difference = next_index - current_index
                        
                        if difference < template_file_len:
                            logging.error(f'Overlap found in matched_event_index_daily (template: {template_file}, day={tmp_i}): '
                                          f'current_index={current_index}, next_index={next_index}, difference={difference}')

In [6]:
# 调用验证函数
verify_overlap(template_len, stations_dir)

INFO - Verifying station: Xs01
INFO - Verifying result file: Xs01_matched_filter_results_threshold0.7_weight0.0_1.0_0.0.h5
INFO - Verifying result file: Xs01_matched_filter_results_threshold0.8_weight0.0_1.0_0.0.h5
INFO - Verifying result file: Xs01_matched_filter_results_threshold0.8_weight0.0_0.0_1.0.h5
INFO - Verifying result file: Xs01_matched_filter_results_threshold0.7_weight0.0_0.0_1.0.h5
INFO - Verifying result file: Xs01_matched_filter_results_threshold0.8_weight1.0_0.0_0.0.h5
INFO - Verifying result file: Xs01_matched_filter_results_threshold0.7_weight1.0_0.0_0.0.h5
INFO - Verifying station: Xs02
INFO - Verifying result file: Xs02_matched_filter_results_threshold0.8_weight0.0_1.0_0.0.h5
INFO - Verifying result file: Xs02_matched_filter_results_threshold0.7_weight0.0_1.0_0.0.h5
INFO - Verifying result file: Xs02_matched_filter_results_threshold0.7_weight0.0_0.0_1.0.h5
INFO - Verifying result file: Xs02_matched_filter_results_threshold0.8_weight0.0_0.0_1.0.h5
INFO - Verifying r

In [12]:
stations = [
"Xs01",
"Xs02",
"Xs03",
"Xs04",
"Xs05",
"Xs06",
"Xs07",
"Xs08",
"Xs09",
"Xs10",
"Xs11",
"Xs12",
"Xs13",
"Xs14",
"Xs16",
"Xs17",
"Xs18",
"Xs19",
"Xs20",
"Xs21",
"Xs22",
"Xs23",
"Xs24",
"Xs25",
"Xs26",
"Xs27",
"Xs28",
"Xs29",
]

for station in stations:
    matched = 0
    data = utils.load_result(f'{station}_matched_filter_results_threshold0.7_weight0.0_0.0_1.0.h5',f'./logs_814-918/{station}/')
    for key in data.keys():
        print(len(data[key]['matched_event_index_daily'])

SyntaxError: unexpected EOF while parsing (<ipython-input-12-9173adb26bf5>, line 36)

In [11]:
print(data.keys())

dict_keys(['template_15021000.h5', 'template_15029300.h5', 'template_15390050.h5', 'template_15625774.h5', 'template_15643044.h5', 'template_15754899.h5', 'template_15992000.h5', 'template_16181250.h5', 'template_16206050.h5', 'template_16229300.h5', 'template_16240899.h5', 'template_16254100.h5', 'template_16254949.h5', 'template_16260300.h5', 'template_16334625.h5', 'template_16334774.h5', 'template_16931250.h5', 'template_17179824.h5', 'template_18725449.h5'])


In [ ]:
template_len

{'template_15021000.h5': 200,
 'template_15029300.h5': 200,
 'template_15390050.h5': 250,
 'template_15625774.h5': 300,
 'template_15643044.h5': 200,
 'template_15754899.h5': 200,
 'template_15992000.h5': 125,
 'template_16181250.h5': 125,
 'template_16206050.h5': 200,
 'template_16229300.h5': 125,
 'template_16240899.h5': 200,
 'template_16254100.h5': 125,
 'template_16254949.h5': 150,
 'template_16260300.h5': 200,
 'template_16334625.h5': 150,
 'template_16334774.h5': 150,
 'template_16931250.h5': 150,
 'template_17179824.h5': 125,
 'template_18725449.h5': 150}

In [3]:
import utils
import numpy as np
stations = [
#"Xs01",
#"Xs02",
#"Xs03",
#"Xs04",
#"Xs05",
#"Xs06",
#"Xs07",
#"Xs08",
#"Xs09",
#"Xs10",
#"Xs11",
#"Xs12",
#"Xs13",
#"Xs14",
#"Xs16",
#"Xs17",
#"Xs18",
#"Xs19",
"Xs20",
"Xs21",
"Xs22",
"Xs23",
"Xs24",
"Xs25",
"Xs26",
"Xs27",
"Xs28",
"Xs29",
]
threshold = '0.8'
weight = '0.0_0.0_1.0'
for station in stations:
	data = utils.load_result(f'{station}_matched_filter_results_threshold{threshold}_weight{weight}.h5',f'./logs_814-918/{station}/')
	matched_count = 0	
	for key in data.keys():
		matched_count += data[key]['matched_count']
	print(station,np.log(matched_count)*1000)

Xs20 7561.6417455887795
Xs21 5846.438775057724
Xs22 8332.308352219117
Xs23 6951.772164398912
Xs24 2197.2245773362197
Xs25 6418.364935936212
Xs26 2944.4389791664403
Xs27 5198.497031265826
Xs28 9790.990521308982
Xs29 7996.653875462607
